In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
# 考虑优化方向：增加层的dropout，全连接层变1*1卷积层
class MultiQueryAttention(nn.Module):
    def __init__(self, heads, dims, attn_scale=None, sr_scale=1):
        super().__init__()
        self.dims = dims
        self.heads = heads
        self.dim_per_head = dims / heads
        
        # 定义获得query、key、value的投影张量
        self.proj_q = nn.Linear(dims, dims * heads)
        self.proj_k = nn.Linear(dims, dims)
        self.proj_v = nn.Linear(dims, dims)
        
        # 考虑下采样
        self.sr_scale = sr_scale
        self.sr = sr_scale
        if sr_scale > 1:
            self.sr = nn.Conv2d(dims, dims, kernel_size=sr_scale, stride=sr_scale)
            self.norm = nn.LayerNorm(dims)
            
        # 考虑注意力分数缩放
        self.scale = attn_scale or self.dim_per_head ** -0.5
        
        # 定义最终投影层，学习多注意力头组合的形式
        self.proj = nn.Conv2d(heads * dims, dims, kernel_size=1, stride=1, bias=True)
        
    def forward(self, x):
        # 得到输入的尺寸(B, C, H, W)
        batch_size, dim, height, width = x.shape
        print(x.shape)
    
        assert height * width == self.sr_scale ** 2 * dim, print("resolution is not compatible with dimension")
        
        # 得到query(B, h, N, C)
        query = self.proj_q(x.reshape(batch_size, -1, dim)).reshape(batch_size, self.heads, -1, dim)
        
        # x = SRA(x) (B, N/4, C)
        x = self.sr(x)
        x = self.norm(x.reshape(batch_size, -1, dim))
        
        # 得到key(B, 1, C, N/4)和value(B, 1, N/4, C)
        key = self.proj_k(x).reshape(batch_size, 1, dim, -1)
        value = self.proj_v(x).reshape(batch_size, 1, -1, dim)
        
        # Q、K相乘，经softmax函数得到注意力分数
        print(query.shape, key.shape)
        attn_score = torch.matmul(query, key) * self.scale
        attn_score = F.softmax(attn_score, dim=-1)

        # value加权，获得最终输出
        logits = torch.matmul(attn_score, value).reshape(batch_size, dim * self.heads, height, width)
        print(logits.shape)
        x = self.proj(logits)
        
        return x.shape

In [3]:
X = torch.ones(2, 16, 8, 8)
net = MultiQueryAttention(heads=4, dims=16, sr_scale=2)
net(X)

torch.Size([2, 16, 8, 8])
torch.Size([2, 4, 64, 16]) torch.Size([2, 1, 16, 16])
torch.Size([2, 64, 8, 8])


torch.Size([2, 16, 8, 8])